# ipynb/umap2.ipynb - UMAP 2 / UMAP 2

## 项目背景 / Background
UMAP 2
UMAP 2

## 功能模块 / Modules
- UMAP 嵌入分析 (2)
- (详见各代码单元 / see code cells)

## 输入 / Inputs
- 上一阶段产物(.npy/.pt/.csv/.json)/ prior-stage outputs
- 内嵌常量与参数 / inline constants and params

## 输出 / Outputs
- 图表(内联显示) / figures (inline)
- 中间变量 / intermediate variables
- 导出文件(.png/.pdf/.csv) / exported files

## 数据流 / Data Flow
1. 加载数据 / Load data
2. 运行分析 / Run analysis
3. 渲染图表 / Render figures
4. 导出 / Export

## 相关文件 / Related Files
- 调用 / Calls: prepare_umap_*.py、ipynb/umap1.ipynb
- 被调用 / Called by: 报告 / 论文 / report / paper

## 使用示例 / Usage Example
- 在 JupyterLab 中打开 / open in JupyterLab
- 逐单元运行 / run cells sequentially

## 作者 / Author
项目组 / Project Team

## 版本 / Version
1.0



In [ ]:
library(readxl)
library(uwot)
library(ggplot2)
library(dplyr)
library(stringr)
library(scales) # 需要加载 scales 包来使用 squish 功能

# --- 1. 定义配色方案 ---
# A. 点的配色 (保持之前的鲜明色系不变)
distinct_colors <- c(
  "#E41A1C", "#377EB8", "#4DAF4A", "#984EA3", "#FF7F00", "#A65628",
  "#F781BF", "#00CED1", "#FF1493", "#32CD32", "#0000FF", "#8B008B" # 把黄色替换掉，防止和核心色冲突
)
label_names_12 <- c("Am", "Atol", "Cm", "Gm", "Tm", "Y", "ac4C", "m1A", "m5C", "m6A", "m6Am", "m7G")
names(distinct_colors) <- label_names_12

# B. 核心密度的单一高光色 (定义一个极明显的颜色，如亮金黄)
core_highlight_color <- "#FFD700" # Gold

# --- 2. 数据处理与采样 ---
df_all <- read_excel("human.xlsx")

str_to_matrix <- function(vec) {
  do.call(rbind, lapply(str_split(str_squish(str_replace_all(vec, "\\[|\\]", "")), "\\s+"), as.numeric))
}

set.seed(42)
df_sampled <- df_all %>%
  group_by(label12) %>%
  sample_n(size = min(n(), 1000)) %>%
  ungroup()

# --- 3. 使用 uwot 计算 UMAP ---
mat_12 <- str_to_matrix(df_sampled$l12)
umap_res <- umap(
  mat_12, 
  n_neighbors = 60,      # 增加邻居数，增强全局聚合
  min_dist = 0.4,        # 增加最小距离，让点分布得更“开”一点以填满空隙
  spread = 0.8,          # 缩小分布跨度，让簇更靠近
  metric = "cosine", 
  n_threads = 16
)

base_df <- data.frame(
  U1 = umap_res[,1],
  U2 = umap_res[,2],
  ActualLabel = label_names_12[df_sampled$label12 + 1]
)

# --- 4. 数据重构 (One-vs-All) ---
expanded_plot_df <- list()
for (target_label in label_names_12) {
  temp_df <- base_df
  temp_df$FacetPanel <- target_label
  temp_df$PlotStatus <- ifelse(temp_df$ActualLabel == target_label, target_label, "Other")
  expanded_plot_df[[target_label]] <- temp_df
}
plot_df_final <- do.call(rbind, expanded_plot_df)
plot_df_final$FacetPanel <- factor(plot_df_final$FacetPanel, levels = label_names_12)

# --- 5. 绘图：调整顺序让金色核心置顶 ---
p <- ggplot(plot_df_final, aes(x = U1, y = U2)) +
  # A. 背景层：最底层，极淡的灰色散点
  geom_point(data = subset(plot_df_final, PlotStatus == "Other"),
             color = "#F0F0F0", size = 0.1, alpha = 0.1) +
  
  # B. 前景层：中间层，实际的分类颜色点
  geom_point(data = subset(plot_df_final, PlotStatus != "Other"),
             aes(color = PlotStatus), 
             size = 0.5, alpha = 0.8) +
  
#   # C. 【关键修改】核心密度层：移动到最后，确保它覆盖在点之上
#   stat_density_2d(
#     data = subset(plot_df_final, PlotStatus != "Other"),
#     aes(alpha = after_stat(nlevel)), 
#     geom = "polygon",
#     fill = core_highlight_color,     # 金色
#     color = NA,                      
#     bins = 15,                       
#     h = c(1.0, 1.0)                  
#   ) +
  
  # C. 前景层：实际的点，保持原来的分类配色
  geom_point(data = subset(plot_df_final, PlotStatus != "Other"),
             aes(color = PlotStatus), # <--- 点的颜色依然映射到类别
             size = 1.5, alpha = 0.8) +
  
  # 配色映射
  scale_color_manual(values = distinct_colors) +
  
  # 【核心技巧】透明度截断控制
  # 解释：nlevel 范围是 0-1。我们设置 limits = c(0.7, 1.0)
  # 意味着只有密度达到最高峰 70% 以上的区域才会被赋予透明度。
  # oob = scales::squish 确保低于 0.7 的区域 alpha 强制为 0 (完全不可见)。
  # range = c(0.5, 0.9) 确保显示出来的核心既有层次感，又足够实。
  scale_alpha_continuous(limits = c(0.7, 1.0), range = c(0.5, 0.9), oob = scales::squish, guide = "none") +
  
  facet_wrap(~FacetPanel, ncol = 4, scales = "fixed") +
  theme_minimal() +
  labs(title = "Visualization of Feature Aggregation Cores",
       subtitle = "Golden Regions Indicate High-Density 'Cores' (>70% Peak Density) vs. Individual Points",
       x = "UMAP Dimension 1", y = "UMAP Dimension 2") +
  theme(
    panel.grid = element_blank(),
    panel.background = element_rect(fill = "white", color = NA),
    strip.text = element_text(face = "bold", size = 12, family = "serif"),
    strip.background = element_rect(fill = "#FAFAFA", color = NA),
    axis.text = element_blank(),
    axis.ticks = element_blank(),
    panel.border = element_rect(color = "grey92", fill = NA, linewidth = 0.3),
    plot.title = element_text(face = "bold", size = 18, hjust = 0.5, family = "serif"),
    plot.subtitle = element_text(hjust = 0.5, color = "grey40", size = 12),
    legend.position = "none"
  )

print(p)

ggsave("UMAP_Decision_Boundary_Morandi.pdf", p, width = 15, height = 11)

In [ ]:
# --- 0. 环境准备 ---
library(reticulate) # 【新增】用于读取 npz
library(uwot)
library(ggplot2)
library(dplyr)
library(scales)
library(tibble)     # 用于更方便的数据框操作

# 注意：请确保你的电脑上安装了 Python 和 numpy
# 如果报错找不到 python，可以使用 use_python("/path/to/python") 指定

# --- 1. 定义配色方案 (保持不变) ---
distinct_colors <- c(
  "#E41A1C", "#377EB8", "#4DAF4A", "#984EA3", "#FF7F00", "#A65628",
  "#F781BF", "#00CED1", "#FF1493", "#32CD32", "#0000FF", "#8B008B"
)
label_names_12 <- c("Am", "Atol", "Cm", "Gm", "Tm", "Y", "ac4C", "m1A", "m5C", "m6A", "m6Am", "m7G")
names(distinct_colors) <- label_names_12

# 核心密度的单一高光色
core_highlight_color <- "#FFD700" 

# --- 2. 数据加载与处理 (【核心修改部分】) ---
use_python("/home/dc/miniconda3/bin/python")
# A. 加载 .npz 数据
np <- import("numpy")
data_npz <- np$load('human_atten.npz')

# B. 提取数据
# 注意：Python索引从0开始，label12 应该是 0-11 的整数
raw_attn <- data_npz$f[['attn_out_12']] # Shape: (203993, 12, 128)
raw_labels <- data_npz$f[['label12']]   # Shape: (203993,)

# C. 数据展平 (Reshape)
# 将 (N, 12, 128) 转换为 (N, 12*128 = 1536) 的 2D 矩阵供 UMAP 使用
# reticulate 的 array_reshape 会自动处理行列优先的内存转换
n_samples <- dim(raw_attn)[1]
n_features <- dim(raw_attn)[2] * dim(raw_attn)[3]
mat_full <- array_reshape(raw_attn, c(n_samples, n_features))

# D. 构建用于采样的元数据框
df_meta <- tibble(
  id = 1:n_samples,
  label_idx = raw_labels # 0-11
)

# E. 采样 (保持每类最多 1000 个)
set.seed(42)
sampled_indices <- df_meta %>%
  group_by(label_idx) %>%
  sample_n(size = min(n(), 5000)) %>%
  ungroup() %>%
  pull(id)

# F. 获取采样后的矩阵和标签
mat_12 <- mat_full[sampled_indices, ]
sampled_labels <- df_meta$label_idx[sampled_indices]

# --- 3. 使用 uwot 计算 UMAP (保持参数不变) ---
print("Running UMAP...")
umap_res <- umap(
  mat_12, 
  n_neighbors = 60,      
  min_dist = 0.4,        
  spread = 0.8,          
  metric = "cosine", 
  n_threads = 16         # 根据你的 CPU 核心数调整
)

# 构建绘图基础数据
# 注意：label_idx 是 0-11，R 索引是从 1 开始，所以需要 +1
base_df <- data.frame(
  U1 = umap_res[,1],
  U2 = umap_res[,2],
  ActualLabel = label_names_12[sampled_labels + 1] 
)

# --- 4. 数据重构 (One-vs-All) ---
expanded_plot_df <- list()
for (target_label in label_names_12) {
  temp_df <- base_df
  temp_df$FacetPanel <- target_label
  temp_df$PlotStatus <- ifelse(temp_df$ActualLabel == target_label, target_label, "Other")
  expanded_plot_df[[target_label]] <- temp_df
}
plot_df_final <- do.call(rbind, expanded_plot_df)
plot_df_final$FacetPanel <- factor(plot_df_final$FacetPanel, levels = label_names_12)

# --- 5. 绘图 ---
p <- ggplot(plot_df_final, aes(x = U1, y = U2)) +
  # A. 背景层：极淡的灰色散点
  geom_point(data = subset(plot_df_final, PlotStatus == "Other"),
             color = "#F0F0F0", size = 0.1, alpha = 0.1) +
  
  # B. 核心密度层 (我帮你取消了注释，这样才能看到金色核心)
  # 注意：这一层必须在背景点之上，但在具体数据点之下或由 alpha 控制
  stat_density_2d(
    data = subset(plot_df_final, PlotStatus != "Other"),
    aes(alpha = after_stat(nlevel)), 
    geom = "polygon",
    fill = core_highlight_color,     
    color = NA,                      
    bins = 15,                       
    h = c(1.0, 1.0) # 如果报错，可以尝试去掉 h 参数让 ggplot 自动计算                 
  ) +
  
  # C. 前景层：实际的点
  geom_point(data = subset(plot_df_final, PlotStatus != "Other"),
             aes(color = PlotStatus), 
             size = 1.5, alpha = 0.8) +
  
  # 配色映射
  scale_color_manual(values = distinct_colors) +
  
  # 透明度控制 (控制金色核心的显示范围)
  scale_alpha_continuous(limits = c(0.7, 1.0), range = c(0.5, 0.9), oob = scales::squish, guide = "none") +
  
  facet_wrap(~FacetPanel, ncol = 4, scales = "fixed") +
  theme_minimal() +
  labs(title = "Visualization of Attention Features (Layer 12)",
       subtitle = "Golden Regions Indicate High-Density 'Cores' (>70% Peak Density)",
       x = "UMAP Dimension 1", y = "UMAP Dimension 2") +
  theme(
    panel.grid = element_blank(),
    panel.background = element_rect(fill = "white", color = NA),
    strip.text = element_text(face = "bold", size = 12, family = "serif"),
    strip.background = element_rect(fill = "#FAFAFA", color = NA),
    axis.text = element_blank(),
    axis.ticks = element_blank(),
    panel.border = element_rect(color = "grey92", fill = NA, linewidth = 0.3),
    plot.title = element_text(face = "bold", size = 18, hjust = 0.5, family = "serif"),
    plot.subtitle = element_text(hjust = 0.5, color = "grey40", size = 12),
    legend.position = "none"
  )

print(p)

ggsave("UMAP_Attn12_Density.pdf", p, width = 15, height = 11)

In [ ]:
# --- 0. 环境准备 ---
library(reticulate)
library(uwot)
library(ggplot2)
library(dplyr)
library(scales)
library(tibble)

# --- 1. 定义参数与配色 ---
# 设定的采样数量 x (正负样本各取 x 个，总共 2x 个点用于跑 UMAP)
target_n_per_class <- 1000  # <--- 你可以在这里修改 x 的大小

distinct_colors <- c(
  "#E41A1C", "#377EB8", "#4DAF4A", "#984EA3", "#FF7F00", "#A65628",
  "#F781BF", "#00CED1", "#FF1493", "#32CD32", "#0000FF", "#8B008B"
)
label_names_12 <- c("Am", "Atol", "Cm", "Gm", "Tm", "Y", "ac4C", "m1A", "m5C", "m6A", "m6Am", "m7G")
names(distinct_colors) <- label_names_12
core_highlight_color <- "#FFD700" # 金色核心

# --- 2. 数据加载与预处理 ---
np <- import("numpy")
data_npz <- np$load('human_atten.npz')

raw_attn <- data_npz$f[['attn_out_12']]
raw_labels <- data_npz$f[['label12']] # 0-11

# 展平数据 (N, 12*128)
n_samples <- dim(raw_attn)[1]
mat_full <- array_reshape(raw_attn, c(n_samples, dim(raw_attn)[2] * dim(raw_attn)[3]))

# --- 3. 循环 12 次：独立采样 + 独立 UMAP ---
plot_list_df <- list()

print(paste("开始处理 12 个子图，每个子图采样正负样本各:", target_n_per_class))

for (i in 0:11) {
  target_name <- label_names_12[i + 1]
  
  # A. 找出正样本和负样本的索引
  pos_indices <- which(raw_labels == i)
  neg_indices <- which(raw_labels != i)
  
  # B. 采样逻辑 (1:1 平衡)
  # 实际取样数：取设定值 x 和实际拥有的最小值
  # (如果某类只有 500 个，那就只取 500 个正样本 + 500 个负样本)
  current_n <- min(length(pos_indices), target_n_per_class)
  
  set.seed(42 + i) # 确保每个循环随机性固定但不同
  sampled_pos <- sample(pos_indices, current_n)
  sampled_neg <- sample(neg_indices, current_n) # 抽取等量的负样本
  
  # 合并索引
  combined_indices <- c(sampled_pos, sampled_neg)
  
  # C. 构建当前子图的特征矩阵和标签
  current_mat <- mat_full[combined_indices, ]
  
  # 标记：Target (正样本) vs Other (负样本)
  status_vec <- c(rep("Target", current_n), rep("Other", current_n))
  
  # D. 运行独立的 UMAP
  # 因为是局部小样本 (2*x)，计算会很快
  cat(sprintf("Running UMAP for %s (Samples: %d)...\n", target_name, length(combined_indices)))
  
  umap_out <- umap(
    current_mat, 
    n_neighbors = 30,  # 样本变少了，稍微减小邻居数以保留局部结构
    min_dist = 0.4,
    metric = "cosine",
    n_threads = 8      # 根据电脑配置调整
  )
  
  # E. 存入列表
  temp_df <- data.frame(
    U1 = umap_out[,1],
    U2 = umap_out[,2],
    PlotStatus = status_vec,      # Target 或 Other
    RealLabel = target_name,      # 这一整张图是属于哪个类别的
    FacetPanel = target_name      # 用于分面
  )
  plot_list_df[[target_name]] <- temp_df
}

# 合并所有数据
plot_df_final <- do.call(rbind, plot_list_df)
plot_df_final$FacetPanel <- factor(plot_df_final$FacetPanel, levels = label_names_12)

# --- 4. 绘图 ---
# 难点：我们需要让每个子图的 "Target" 点显示该类别对应的颜色，"Other" 显示灰色
# 这里使用一个小技巧：在 aes 中动态映射颜色

p <- ggplot(plot_df_final, aes(x = U1, y = U2)) +
  
  # A. 负样本层 (背景层)
  geom_point(data = subset(plot_df_final, PlotStatus == "Other"),
             color = "grey85", size = 0.3, alpha = 0.3) +
  
  # B. 金色核心密度层 (只针对 Target 正样本计算)
  stat_density_2d(
    data = subset(plot_df_final, PlotStatus == "Target"),
    aes(alpha = after_stat(nlevel)), 
    geom = "polygon",
    fill = core_highlight_color,     
    color = NA,                      
    bins = 10,                       
    h = c(1.5, 1.5) # 稍微调大带宽，因为样本可能变稀疏了
  ) +
  
  # C. 正样本层 (前景层)
  # 这里为了让不同分面用不同颜色，我们需要把 FacetPanel 映射给 color
  geom_point(data = subset(plot_df_final, PlotStatus == "Target"),
             aes(color = FacetPanel), 
             size = 1.0, alpha = 0.8) +
  
  # 颜色映射：让颜色名与 FacetPanel 对应
  scale_color_manual(values = distinct_colors) +
  
  # 透明度控制 (密度层)
  scale_alpha_continuous(limits = c(0.6, 1.0), range = c(0.4, 0.8), oob = scales::squish, guide = "none") +
  
  # 分面设置
  # 【关键】scales = "free" 是必须的！
  # 因为我们跑了 12 次独立的 UMAP，每次的坐标范围完全不同，不能固定坐标轴
  facet_wrap(~FacetPanel, ncol = 4, scales = "free") +
  
  theme_minimal() +
  labs(title = "One-vs-Rest Balanced UMAP Projection",
       subtitle = sprintf("Each panel is an independent UMAP of 'Target' (n=%d) vs Balanced 'Others' (n=%d)", target_n_per_class, target_n_per_class),
       x = "UMAP Dim 1", y = "UMAP Dim 2") +
  theme(
    panel.grid = element_blank(),
    panel.background = element_rect(fill = "white", color = NA),
    strip.text = element_text(face = "bold", size = 12),
    strip.background = element_rect(fill = "#FAFAFA", color = NA),
    axis.text = element_blank(),
    axis.ticks = element_blank(),
    panel.border = element_rect(color = "grey92", fill = NA, linewidth = 0.3),
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    legend.position = "none" # 不需要图例，标题已经说明了
  )

print(p)
ggsave("UMAP_Independent_Balanced.pdf", p, width = 15, height = 11)

In [ ]:
# --- 0. 环境准备 ---
library(reticulate)
library(uwot)
library(ggplot2)
library(dplyr)
library(scales)
library(tibble)

# --- 1. 定义参数与配色 ---
# 设定的采样数量 x (正负样本各取 x 个，总共 2x 个点用于跑 UMAP)
target_n_per_class <- 5000  # <--- 你可以在这里修改 x 的大小

distinct_colors <- c(
  "#E41A1C", "#377EB8", "#4DAF4A", "#984EA3", "#FF7F00", "#A65628",
  "#F781BF", "#00CED1", "#FF1493", "#32CD32", "#0000FF", "#8B008B"
)
label_names_12 <- c("Am", "Atol", "Cm", "Gm", "Tm", "Y", "ac4C", "m1A", "m5C", "m6A", "m6Am", "m7G")
names(distinct_colors) <- label_names_12
core_highlight_color <- "#FFD700" # 金色核心

# --- 2. 数据加载与预处理 ---
np <- import("numpy")
data_npz <- np$load('human_atten.npz')

raw_attn <- data_npz$f[['attn_out_12']]
raw_labels <- data_npz$f[['label12']] # 0-11

# 展平数据 (N, 12*128)
n_samples <- dim(raw_attn)[1]
mat_full <- array_reshape(raw_attn, c(n_samples, dim(raw_attn)[2] * dim(raw_attn)[3]))

# --- 3. 循环 12 次：独立采样 + 独立 UMAP ---
plot_list_df <- list()

print(paste("开始处理 12 个子图，每个子图采样正负样本各:", target_n_per_class))

for (i in 0:11) {
  target_name <- label_names_12[i + 1]
  
  # A. 找出正样本和负样本的索引
  pos_indices <- which(raw_labels == i)
  neg_indices <- which(raw_labels != i)
  
  # B. 采样逻辑 (1:1 平衡)
  # 实际取样数：取设定值 x 和实际拥有的最小值
  # (如果某类只有 500 个，那就只取 500 个正样本 + 500 个负样本)
  current_n <- min(length(pos_indices), target_n_per_class)
  
  set.seed(42 + i) # 确保每个循环随机性固定但不同
  sampled_pos <- sample(pos_indices, current_n)
  sampled_neg <- sample(neg_indices, current_n) # 抽取等量的负样本
  
  # 合并索引
  combined_indices <- c(sampled_pos, sampled_neg)
  
  # C. 构建当前子图的特征矩阵和标签
  current_mat <- mat_full[combined_indices, ]
  
  # 标记：Target (正样本) vs Other (负样本)
  status_vec <- c(rep("Target", current_n), rep("Other", current_n))
  
  # D. 运行独立的 UMAP
  # 因为是局部小样本 (2*x)，计算会很快
  cat(sprintf("Running UMAP for %s (Samples: %d)...\n", target_name, length(combined_indices)))
  
  umap_out <- umap(
    current_mat, 
    n_neighbors = 30,  # 样本变少了，稍微减小邻居数以保留局部结构
    min_dist = 0.4,
    metric = "cosine",
    n_threads = 8      # 根据电脑配置调整
  )
  
  # E. 存入列表
  temp_df <- data.frame(
    U1 = umap_out[,1],
    U2 = umap_out[,2],
    PlotStatus = status_vec,      # Target 或 Other
    RealLabel = target_name,      # 这一整张图是属于哪个类别的
    FacetPanel = target_name      # 用于分面
  )
  plot_list_df[[target_name]] <- temp_df
}

# 合并所有数据
plot_df_final <- do.call(rbind, plot_list_df)
plot_df_final$FacetPanel <- factor(plot_df_final$FacetPanel, levels = label_names_12)

# --- 4. 绘图 ---
# 难点：我们需要让每个子图的 "Target" 点显示该类别对应的颜色，"Other" 显示灰色
# 这里使用一个小技巧：在 aes 中动态映射颜色

p <- ggplot(plot_df_final, aes(x = U1, y = U2)) +
  
  # A. [最底层] 负样本 (背景，灰色)
  geom_point(data = subset(plot_df_final, PlotStatus == "Other"),
             color = "grey90", size = 0.5, alpha = 0.3) +
  
  # B. [中间层] 正样本 (前景，彩色点)
  # 注意：点现在放在密度图下面了
  geom_point(data = subset(plot_df_final, PlotStatus == "Target"),
             aes(color = FacetPanel), 
             size = 1.2, alpha = 0.6) + # 稍微降低点的透明度，以免干扰上面的金色
  
  # C. [最顶层] 金色核心密度层 (覆盖在点上)
  stat_density_2d(
    data = subset(plot_df_final, PlotStatus == "Target"),
    aes(alpha = after_stat(nlevel)), # 透明度映射密度
    geom = "polygon",
    fill = "#FFD700",                # 纯亮金色
    color = NA,                      # 不要默认的多边形边框，太乱
    bins = 8,                        # 减少层级数，让块面更整洁
    h = c(1.2, 1.2)                  # 平滑度带宽
  ) +
  
  # D. [增强边界] 额外加一条清晰的金色轮廓线
  stat_density_2d(
    data = subset(plot_df_final, PlotStatus == "Target"),
    geom = "path",                   # 仅画线
    color = "#B8860B",               # 深金色 (DarkGoldenRod) 做边框，增加立体感
    size = 0.3,
    bins = 8,
    h = c(1.2, 1.2),
    alpha = 0.6
  ) +
  
  # 颜色映射 (对应点的颜色)
  scale_color_manual(values = distinct_colors) +
  
  # 【核心修改】透明度控制：让金色更明显
  scale_alpha_continuous(
    limits = c(0.2, 1.0),     # 只有密度大于 20% 的区域才显示 (过滤掉背景噪音)
    range = c(0.3, 0.85),     # 显示范围：最淡的地方 0.3，最浓的地方 0.85 (很实)
    oob = scales::squish,     # 低于 0.2 的强制设为透明
    guide = "none"
  ) +
  
  facet_wrap(~FacetPanel, ncol = 4, scales = "free") +
  
  theme_minimal() +
  labs(title = "Feature Density Cores (Gold Layer Overlay)",
       subtitle = "Gold regions indicate high-confidence aggregation areas, overlaying individual points.",
       x = "UMAP Dim 1", y = "UMAP Dim 2") +
  theme(
    panel.grid = element_blank(),
    panel.background = element_rect(fill = "white", color = NA),
    strip.text = element_text(face = "bold", size = 12),
    strip.background = element_rect(fill = "#FAFAFA", color = NA),
    axis.text = element_blank(),
    axis.ticks = element_blank(),
    panel.border = element_rect(color = "grey92", fill = NA, linewidth = 0.3),
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    legend.position = "none"
  )

print(p)
ggsave("UMAP_Gold_Overlay.pdf", p, width = 15, height = 11)

In [ ]:
# --- 0. 环境准备 ---
library(reticulate)
library(uwot)
library(ggplot2)
library(dplyr)
library(scales)
library(tibble)

# --- 1. 定义参数与配色 ---
# 设定的采样数量 x (正负样本各取 x 个，总共 2x 个点用于跑 UMAP)
target_n_per_class <- 5000  # <--- 你可以在这里修改 x 的大小

distinct_colors <- c(
  "#E41A1C", "#377EB8", "#4DAF4A", "#984EA3", "#FF7F00", "#A65628",
  "#F781BF", "#00CED1", "#FF1493", "#32CD32", "#0000FF", "#8B008B"
)
label_names_12 <- c("Am", "Atol", "Cm", "Gm", "Tm", "Y", "ac4C", "m1A", "m5C", "m6A", "m6Am", "m7G")
names(distinct_colors) <- label_names_12
core_highlight_color <- "#FFD700" # 金色核心

# --- 2. 数据加载与预处理 ---
np <- import("numpy")
data_npz <- np$load('human_atten.npz')

raw_attn <- data_npz$f[['attn_out_12']]
raw_labels <- data_npz$f[['label12']] # 0-11

# 展平数据 (N, 12*128)
n_samples <- dim(raw_attn)[1]
mat_full <- array_reshape(raw_attn, c(n_samples, dim(raw_attn)[2] * dim(raw_attn)[3]))

# --- 3. 循环 12 次：独立采样 + 独立 UMAP ---
plot_list_df <- list()

print(paste("开始处理 12 个子图，每个子图采样正负样本各:", target_n_per_class))

for (i in 0:11) {
  target_name <- label_names_12[i + 1]
  pos_indices <- which(raw_labels == i)
  neg_indices <- which(raw_labels != i)
  
  current_n <- min(length(pos_indices), target_n_per_class)
  
  # --- 核心：最远负样本筛选 ---
  # 计算正样本中心
  pos_feat <- mat_full[pos_indices, ]
  pos_feat_norm <- pos_feat / sqrt(rowSums(pos_feat^2) + 1e-9)
  centroid <- colMeans(pos_feat_norm)
  centroid <- centroid / sqrt(sum(centroid^2) + 1e-9)
  
  # 计算负样本到中心的相似度
  neg_feat <- mat_full[neg_indices, ]
  neg_feat_norm <- neg_feat / sqrt(rowSums(neg_feat^2) + 1e-9)
  cos_sim <- as.vector(neg_feat_norm %*% matrix(centroid, ncol=1))
  
  # 选取最不相似的
  sampled_pos <- sample(pos_indices, current_n) 
  sampled_neg <- neg_indices[order(cos_sim)[1:current_n]]
  
  combined_indices <- c(sampled_pos, sampled_neg)
  current_mat <- mat_full[combined_indices, ]
  status_vec <- c(rep("Target", length(sampled_pos)), rep("Other", length(sampled_neg)))
  
  # --- 后续 UMAP 逻辑不变 ---
  cat(sprintf("Running UMAP for %s with Most-Distinct Negatives...\n", target_name))
  # ... [umap 运算及绘图逻辑] ...
  
#   umap_out <- umap(
#     current_mat, 
#     n_neighbors = 30,  # 样本变少了，稍微减小邻居数以保留局部结构
#     min_dist = 0.4,
#     metric = "cosine",
#     n_threads = 8      # 根据电脑配置调整
#   )
umap_out <- umap(
    current_mat, 
    n_neighbors = 100,  # 样本变少了，稍微减小邻居数以保留局部结构
    min_dist = 1.0,
    metric = "cosine",
    n_threads = 64      # 根据电脑配置调整
  )
  
  # E. 存入列表
  temp_df <- data.frame(
    U1 = umap_out[,1],
    U2 = umap_out[,2],
    PlotStatus = status_vec,      # Target 或 Other
    RealLabel = target_name,      # 这一整张图是属于哪个类别的
    FacetPanel = target_name      # 用于分面
  )
  plot_list_df[[target_name]] <- temp_df
}

# 合并所有数据
plot_df_final <- do.call(rbind, plot_list_df)
plot_df_final$FacetPanel <- factor(plot_df_final$FacetPanel, levels = label_names_12)

# --- 4. 绘图 ---
# 难点：我们需要让每个子图的 "Target" 点显示该类别对应的颜色，"Other" 显示灰色
# 这里使用一个小技巧：在 aes 中动态映射颜色

p <- ggplot(plot_df_final, aes(x = U1, y = U2)) +
  
  # A. [最底层] 负样本 (背景，灰色)
  geom_point(data = subset(plot_df_final, PlotStatus == "Other"),
             color = "grey90", size = 0.5, alpha = 0.3) +
  
  # B. [中间层] 正样本 (前景，彩色点)
  # 注意：点现在放在密度图下面了
  geom_point(data = subset(plot_df_final, PlotStatus == "Target"),
             aes(color = FacetPanel), 
             size = 1.2, alpha = 1.0) + # 稍微降低点的透明度，以免干扰上面的金色
  
  # C. [最顶层] 金色核心密度层 (覆盖在点上)
  stat_density_2d(
    data = subset(plot_df_final, PlotStatus == "Target"),
    aes(alpha = after_stat(nlevel)), # 透明度映射密度
    geom = "polygon",
    fill = "#FFD700",                # 纯亮金色
    color = NA,                      # 不要默认的多边形边框，太乱
    bins = 8,                        # 减少层级数，让块面更整洁
    h = c(1.2, 1.2),                  # 平滑度带宽
    alpha = 0.4
  ) +
  
  # D. [增强边界] 额外加一条清晰的金色轮廓线
  stat_density_2d(
    data = subset(plot_df_final, PlotStatus == "Target"),
    geom = "path",                   # 仅画线
    color = "#B8860B",               # 深金色 (DarkGoldenRod) 做边框，增加立体感
    size = 0.3,
    bins = 8,
    h = c(1.2, 1.2),
    alpha = 0.4
  ) +
  
  # 颜色映射 (对应点的颜色)
  scale_color_manual(values = distinct_colors) +
  
  # 【核心修改】透明度控制：让金色更明显
  scale_alpha_continuous(
    limits = c(0.2, 1.0),     # 只有密度大于 20% 的区域才显示 (过滤掉背景噪音)
    range = c(0.3, 0.85),     # 显示范围：最淡的地方 0.3，最浓的地方 0.85 (很实)
    oob = scales::squish,     # 低于 0.2 的强制设为透明
    guide = "none"
  ) +
  
  facet_wrap(~FacetPanel, ncol = 4, scales = "free") +
  
  theme_minimal() +
  labs(title = "Feature Density Cores (Gold Layer Overlay)",
       subtitle = "Gold regions indicate high-confidence aggregation areas, overlaying individual points.",
       x = "UMAP Dim 1", y = "UMAP Dim 2") +
  theme(
    panel.grid = element_blank(),
    panel.background = element_rect(fill = "white", color = NA),
    strip.text = element_text(face = "bold", size = 12),
    strip.background = element_rect(fill = "#FAFAFA", color = NA),
    axis.text = element_blank(),
    axis.ticks = element_blank(),
    panel.border = element_rect(color = "grey92", fill = NA, linewidth = 0.3),
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    legend.position = "none"
  )

print(p)
ggsave("UMAP_Gold_Overlay2.pdf", p, width = 15, height = 11)

In [ ]:
# --- 0. 环境准备 ---
library(reticulate)
library(uwot)
library(ggplot2)
library(dplyr)
library(scales)
library(tibble)

# --- 1. 莫兰迪配色定义 ---
morandi_colors <- c(
  "Am" = "#B9837D", "Atol" = "#7A9CC6", "Cm" = "#8FA68E", 
  "Gm" = "#A090B8", "Tm" = "#D4B082", "Y" = "#B8A08C",
  "ac4C" = "#D6A3B8", "m1A" = "#7DB5B5", "m5C" = "#9BB89C", 
  "m6A" = "#B89595", "m6Am" = "#A3B8C7", "m7G" = "#B8A8C5"
)
core_champagne <- "#D4C5A0"
morandi_grey <- "#C8C4C0"

# 需要先定义 label_names_12
label_names_12 <- c("Am", "Atol", "Cm", "Gm", "Tm", "Y", "ac4C", "m1A", "m5C", "m6A", "m6Am", "m7G")

# --- 2. 数据加载与预处理 ---
np <- import("numpy")
data_npz <- np$load('data/human_atten.npz')

raw_attn <- data_npz$f[['attn_out_12']]
raw_labels <- data_npz$f[['label12']] # 0-11

# 展平数据 (N, 12*128)
n_samples <- dim(raw_attn)[1]
mat_full <- array_reshape(raw_attn, c(n_samples, dim(raw_attn)[2] * dim(raw_attn)[3]))

# --- 3. 循环 12 次：独立采样 + 独立 UMAP ---
plot_list_df <- list()
target_n_per_class <- 5000

print(paste("开始处理 12 个子图，每个子图采样正负样本各:", target_n_per_class))

for (i in 0:11) {
  target_name <- label_names_12[i + 1]
  pos_indices <- which(raw_labels == i)
  neg_indices <- which(raw_labels != i)
  
  current_n <- min(length(pos_indices), target_n_per_class)
  
  # --- 核心：最远负样本筛选 ---
  # 计算正样本中心
  pos_feat <- mat_full[pos_indices, ]
  pos_feat_norm <- pos_feat / sqrt(rowSums(pos_feat^2) + 1e-9)
  centroid <- colMeans(pos_feat_norm)
  centroid <- centroid / sqrt(sum(centroid^2) + 1e-9)
  
  # 计算负样本到中心的相似度
  neg_feat <- mat_full[neg_indices, ]
  neg_feat_norm <- neg_feat / sqrt(rowSums(neg_feat^2) + 1e-9)
  cos_sim <- as.vector(neg_feat_norm %*% matrix(centroid, ncol=1))
  
  # 选取最不相似的
  sampled_pos <- sample(pos_indices, current_n) 
  sampled_neg <- neg_indices[order(cos_sim)[1:current_n]]
  
  combined_indices <- c(sampled_pos, sampled_neg)
  current_mat <- mat_full[combined_indices, ]
  status_vec <- c(rep("Target", length(sampled_pos)), rep("Other", length(sampled_neg)))
  
  # --- 后续 UMAP 逻辑不变 ---
  cat(sprintf("Running UMAP for %s with Most-Distinct Negatives...\n", target_name))
  
  umap_out <- umap(
    current_mat, 
    n_neighbors = 100,
    min_dist = 1.0,
    metric = "cosine",
    n_threads = 64
  )
  
  # E. 存入列表
  temp_df <- data.frame(
    U1 = umap_out[,1],
    U2 = umap_out[,2],
    PlotStatus = status_vec,
    RealLabel = target_name,
    FacetPanel = target_name
  )
  plot_list_df[[target_name]] <- temp_df
}

# 合并所有数据
plot_df_final <- do.call(rbind, plot_list_df)
plot_df_final$FacetPanel <- factor(plot_df_final$FacetPanel, levels = label_names_12)

# --- 4. 绘图 ---
p <- ggplot(plot_df_final, aes(x = U1, y = U2)) +
  
  # A. 负样本（最底层）
  geom_point(data = ~ subset(.x, PlotStatus == "Other"),
             color = morandi_grey, size = 0.4, alpha = 0.25) +
  
  # B. 正样本（中间层）
  geom_point(data = ~ subset(.x, PlotStatus == "Target"),
             aes(color = FacetPanel), 
             size = 1.2, alpha = 0.9, stroke = 0.15) +
  
  # C. 香槟金密度层（注意：alpha只能在一个地方设置）
  stat_density_2d(
    data = ~ subset(.x, PlotStatus == "Target"),
    aes(alpha = after_stat(nlevel)),  # alpha在这里通过aes控制
    geom = "polygon",
    fill = core_champagne,
    color = NA,
    bins = 10,
    h = c(1.5, 1.5)
    # 移除了alpha = 0.35，避免冲突
  ) +
  
  # D. 密度轮廓线
  stat_density_2d(
    data = ~ subset(.x, PlotStatus == "Target"),
    geom = "path",
    color = "#B8A37F",
    size = 0.25,
    bins = 10,
    h = c(1.5, 1.5),
    alpha = 0.8  # 这里alpha是独立的，可以保留
  ) +
  
  # 颜色与透明度控制
  scale_color_manual(values = morandi_colors) +
  scale_alpha_continuous(
    limits = c(0.23, 1.0), 
    range = c(0.2, 0.7), 
    oob = scales::squish, 
    guide = "none"
  ) +
  
  facet_wrap(~ FacetPanel, ncol = 4, scales = "free") +
  
  theme_minimal() +
  labs(
    title = "Methylation Feature Distribution (Morandi Palette)",
    subtitle = "Champagne-gold cores highlight high-density regions",
    x = "UMAP Dim 1", 
    y = "UMAP Dim 2"
  ) +  # 移除了错误的 ... 占位符
  theme(
    panel.grid = element_blank(),
    plot.background = element_rect(fill = "#F9F8F7", color = NA),
    panel.background = element_rect(fill = "#F5F4F2", color = NA),
    strip.text = element_text(face = "bold", size = 12, color = "#4A4A4A"),
    strip.background = element_rect(fill = "#E8E6E3", color = "#C8C4C0", linewidth = 0.4),
    axis.text = element_blank(),
    axis.ticks = element_blank(),
    panel.border = element_rect(color = "grey92", fill = NA, linewidth = 0.3),
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    legend.position = "none"
  )

print(p)
ggsave("png/UMAP_Morandi_Final.pdf", p, width = 15, height = 11, device = cairo_pdf)


In [ ]:
# install.packages("plotly")

In [ ]:
# # --- 0. 环境准备 ---
# library(reticulate)
# library(uwot)
# library(ggplot2)
# library(dplyr)
# library(plotly)    # 改用plotly进行3D渲染
# library(htmltools) # 用于整合多个3D图

# # 配色方案保持不变
# morandi_colors <- c(
#   "Am" = "#B9837D", "Atol" = "#7A9CC6", "Cm" = "#8FA68E", 
#   "Gm" = "#A090B8", "Tm" = "#D4B082", "Y" = "#B8A08C",
#   "ac4C" = "#D6A3B8", "m1A" = "#7DB5B5", "m5C" = "#9BB89C", 
#   "m6A" = "#B89595", "m6Am" = "#A3B8C7", "m7G" = "#B8A8C5"
# )
# core_champagne <- "#D4C5A0"
# morandi_grey <- "#C8C4C0"

# label_names_12 <- c("Am", "Atol", "Cm", "Gm", "Tm", "Y", 
#                     "ac4C", "m1A", "m5C", "m6A", "m6Am", "m7G")

# # --- 1. 数据加载（同原代码） ---
# np <- import("numpy")
# data_npz <- np$load('data/human_atten.npz')
# raw_attn <- data_npz$f[['attn_out_12']]
# raw_labels <- data_npz$f[['label12']]

# n_samples <- dim(raw_attn)[1]
# mat_full <- array_reshape(raw_attn, c(n_samples, dim(raw_attn)[2] * dim(raw_attn)[3]))

# # --- 2. 循环生成3D UMAP ---
# plotly_list <- list()
# target_n_per_class <- 5000

# for (i in 0:11) {
#   target_name <- label_names_12[i + 1]
#   print(paste("处理 3D UMAP:", target_name))
  
#   # 采样逻辑（同原代码）
#   pos_indices <- which(raw_labels == i)
#   neg_indices <- which(raw_labels != i)
#   current_n <- min(length(pos_indices), target_n_per_class)
  
#   # 最远负样本筛选
#   pos_feat <- mat_full[pos_indices, ]
#   pos_feat_norm <- pos_feat / sqrt(rowSums(pos_feat^2) + 1e-9)
#   centroid <- colMeans(pos_feat_norm)
#   centroid <- centroid / sqrt(sum(centroid^2) + 1e-9)
  
#   neg_feat <- mat_full[neg_indices, ]
#   neg_feat_norm <- neg_feat / sqrt(rowSums(neg_feat^2) + 1e-9)
#   cos_sim <- as.vector(neg_feat_norm %*% matrix(centroid, ncol=1))
  
#   sampled_pos <- sample(pos_indices, current_n) 
#   sampled_neg <- neg_indices[order(cos_sim)[1:current_n]]
#   combined_indices <- c(sampled_pos, sampled_neg)
#   current_mat <- mat_full[combined_indices, ]
  
#   # --- 核心修改：生成3D坐标 ---
#   umap_3d <- umap(
#     current_mat,
#     n_components = 3,  # 改为3维！
#     n_neighbors = 100,
#     min_dist = 1.0,
#     metric = "cosine",
#     n_threads = 64
#   )
  
#   # 准备plotly数据
#   status_vec <- c(rep("Target", length(sampled_pos)), 
#                   rep("Other", length(sampled_neg)))
  
#   df_3d <- data.frame(
#     X = umap_3d[,1],
#     Y = umap_3d[,2], 
#     Z = umap_3d[,3],
#     Status = factor(status_vec, levels = c("Other", "Target")),
#     ColorGroup = target_name
#   )
  
#   # 准备数据框（增加颜色和透明度列）
#   df_3d <- data.frame(
#     X = umap_3d[,1],
#     Y = umap_3d[,2], 
#     Z = umap_3d[,3],
#     Status = c(rep("Target", length(sampled_pos)), 
#                rep("Other", length(sampled_neg)))
#   )
  
#   # --- 关键修复：分层添加标记点 ---
#   p_3d <- plot_ly() %>%
    
#     # A. 底层：Other类（半透明灰色）
#     add_markers(
#       data = df_3d %>% filter(Status == "Other"),
#       x = ~X, y = ~Y, z = ~Z,
#       color = I(morandi_grey),  # I()表示静态颜色
#       opacity = 0.15,
#       size = 1,
#       marker = list(line = list(width = 0)),  # 无边框
#       name = "Other",
#       hoverinfo = "skip"  # 不显示悬停信息
#     ) %>%
    
#     # B. 顶层：Target类（实心彩色）
#     add_markers(
#       data = df_3d %>% filter(Status == "Target"),
#       x = ~X, y = ~Y, z = ~Z,
#       color = I(morandi_colors[target_name]),
#       opacity = 0.85,
#       size = 3,
#       marker = list(line = list(width = 0.2, color = "white")),  # 轻微白边
#       name = target_name,
#       hovertemplate = paste0("<b>", target_name, "</b><br>",
#                             "UMAP1: %{x:.2f}<br>",
#                             "UMAP2: %{y:.2f}<br>",
#                             "UMAP3: %{z:.2f}<br>")
#     ) %>%
    
#     # C. 布局设置
#     layout(
#       title = paste0("<b>", target_name, "</b> 3D Feature Distribution"),
#       scene = list(
#         xaxis = list(title = "UMAP1", showgrid = FALSE, zeroline = FALSE, 
#                      color = "#8A8A8A"),
#         yaxis = list(title = "UMAP2", showgrid = FALSE, zeroline = FALSE,
#                      color = "#8A8A8A"),
#         zaxis = list(title = "UMAP3", showgrid = FALSE, zeroline = FALSE,
#                      color = "#8A8A8A"),
#         bgcolor = "#F5F4F2",
#         camera = list(eye = list(x = 1.5, y = 1.5, z = 1.5))  # 默认视角
#       ),
#       font = list(family = "Arial", size = 12, color = "#4A4A4A"),
#       paper_bgcolor = "#F9F8F7",
#       margin = list(l = 0, r = 0, b = 0, t = 50)  # 减小边距
#     ) %>%
    
#     # 隐藏图例（保持简洁）
#     hide_legend()  # 不是hide_colorbar()!
  
#   plotly_list[[target_name]] <- p_3d
# }

# # --- 3. 生成交互式HTML报告 ---
# htmltags <- lapply(names(plotly_list), function(name) {
#   tagList(
#     tags$h2(name, style = "color: #4A4A4A; text-align: center;"),
#     tags$div(
#       style = "display: flex; justify-content: center; margin-bottom: 40px;",
#       plotly_list[[name]]
#     )
#   )
# })

# # 保存为单个HTML文件
# save_html(htmltags, "UMAP_3D_Morandi_Report.html")

# print("✅ 3D UMAP报告已生成！请在浏览器中打开 UMAP_3D_Morandi_Report.html")


In [ ]:
# --- 0. 环境准备 ---
library(reticulate)
library(uwot)
library(ggplot2)
library(dplyr)
library(scales)
library(tibble)
library(cluster)  # 计算轮廓系数
library(FNN)      # 快速计算 k-NN

# --- 1. 莫兰迪配色定义 ---
morandi_colors <- c(
  "Am" = "#B9837D", "Atol" = "#7A9CC6", "Cm" = "#8FA68E", 
  "Gm" = "#A090B8", "Tm" = "#D4B082", "Y" = "#B8A08C",
  "ac4C" = "#D6A3B8", "m1A" = "#7DB5B5", "m5C" = "#9BB89C", 
  "m6A" = "#B89595", "m6Am" = "#A3B8C7", "m7G" = "#B8A8C5"
)
core_champagne <- "#D4C5A0"
morandi_grey <- "#C8C4C0"

label_names_12 <- c("Am", "Atol", "Cm", "Gm", "Tm", "Y", "ac4C", "m1A", "m5C", "m6A", "m6Am", "m7G")

# --- 2. 数据加载与预处理 ---
np <- import("numpy")
data_npz <- np$load('data/human_atten.npz')

# 加载特征 (N, 12, 128)
raw_attn <- data_npz$f[['attn_out_12']]
# 加载标签 (N, 12) -> 这里是关键，它是矩阵
raw_labels <- data_npz$f[['label12']] 

# 1. 展平特征 (N, 12*128 = 1536)
n_samples_attn <- dim(raw_attn)[1]
mat_full <- array_reshape(raw_attn, c(n_samples_attn, dim(raw_attn)[2] * dim(raw_attn)[3]))

# 2. 维度检查与对齐
# 注意：raw_labels 现在是矩阵，我们要检查 nrow
n_samples_label <- dim(raw_labels)[1]

cat(sprintf("原始维度检查:\n特征矩阵: %d 行\n标签矩阵: %d 行\n", nrow(mat_full), n_samples_label))

# 取两者行数的最小值
min_len <- min(nrow(mat_full), n_samples_label)

if (nrow(mat_full) != n_samples_label) {
  cat(sprintf("⚠️ 警告：行数不匹配！正在截断对齐至 %d 行。\n", min_len))
  mat_full <- mat_full[1:min_len, ]
  # 对于矩阵，使用逗号截取行
  raw_labels <- raw_labels[1:min_len, ] 
} else {
  cat("✅ 维度对齐检查通过。\n")
}

# --- 3. 循环 12 次：处理与计算 ---
plot_list_df <- list()

# 参数设置
target_n_per_class <- Inf
k_neighbors_purity <- 50   

print(paste("开始处理 12 个类别... target_n =", target_n_per_class))

for (i in 0:11) {
  target_name <- label_names_12[i + 1]
  
  # --- 【核心修改】针对 (N, 12) 标签的处理 ---
  # Python索引 i 对应 R索引 i+1
  # 取出第 i 列：1表示属于该类(Pos)，0表示不属于(Neg)
  current_col_labels <- raw_labels[, i + 1]
  
  pos_indices <- which(current_col_labels == 1)
  neg_indices <- which(current_col_labels == 0)
  
  # --- 数量控制逻辑 ---
  if (is.infinite(target_n_per_class)) {
    current_n <- length(pos_indices)
  } else {
    current_n <- min(length(pos_indices), target_n_per_class)
  }
  
  # 打印当前类别的样本分布情况
  cat(sprintf("[%s] 总正样本: %d, 总负样本: %d. 计划采样: %d\n", 
              target_name, length(pos_indices), length(neg_indices), current_n))

  # --- 最远负样本筛选 ---
  # 计算正样本中心
  pos_feat <- mat_full[pos_indices, , drop=FALSE] 
  pos_feat_norm <- pos_feat / sqrt(rowSums(pos_feat^2) + 1e-9)
  centroid <- colMeans(pos_feat_norm)
  centroid <- centroid / sqrt(sum(centroid^2) + 1e-9)
  
  # 计算负样本到中心的相似度
  neg_feat <- mat_full[neg_indices, , drop=FALSE]
  neg_feat_norm <- neg_feat / sqrt(rowSums(neg_feat^2) + 1e-9)
  cos_sim <- as.vector(neg_feat_norm %*% matrix(centroid, ncol=1))
  
  # 采样
  sampled_pos <- if(length(pos_indices) > current_n) sample(pos_indices, current_n) else pos_indices
  
  if(length(neg_indices) >= current_n) {
    sampled_neg <- neg_indices[order(cos_sim)[1:current_n]]
  } else {
    sampled_neg <- neg_indices 
  }
  
  combined_indices <- c(sampled_pos, sampled_neg)
  current_mat <- mat_full[combined_indices, , drop=FALSE]
  
  # 标签构造
  labels_numeric <- c(rep(1, length(sampled_pos)), rep(0, length(sampled_neg)))
  status_vec <- c(rep("Target", length(sampled_pos)), rep("Other", length(sampled_neg)))
  
  # --- UMAP ---
  cat(sprintf("   Running UMAP...\n"))
  
  umap_out <- umap(
    current_mat, 
    n_neighbors = 100,
    min_dist = 1.0,
    metric = "cosine",
    n_threads = 64
  )
  
  # --- 计算指标 ---
  
  # 1. 轮廓系数 (Silhouette Coefficient)
  if(length(unique(labels_numeric)) > 1 && nrow(umap_out) > 2) {
    dist_matrix <- dist(umap_out)
    sil_obj <- cluster::silhouette(labels_numeric, dist_matrix)
    sil_score <- mean(sil_obj[, 3])
  } else {
    sil_score <- 0
  }
  
  # 2. k-NN Purity (k=50)
  knn_k <- min(k_neighbors_purity, nrow(umap_out) - 1)
  knn_res <- FNN::get.knn(umap_out, k = knn_k)
  knn_idx <- knn_res$nn.index
  
  neighbor_labels <- matrix(labels_numeric[knn_idx], ncol = knn_k)
  is_match <- neighbor_labels == labels_numeric
  purity_score <- sum(is_match) / (nrow(umap_out) * knn_k)
  
  cat(sprintf("   > Metrics: SC=%.3f, Purity=%.3f\n", sil_score, purity_score))
  
  # facet_label_str <- sprintf("%s\nSC: %.2f | Purity: %.2f", target_name, sil_score, purity_score)
  
  # E. 存入列表
  temp_df <- data.frame(
    U1 = umap_out[,1],
    U2 = umap_out[,2],
    PlotStatus = status_vec,
    RealLabel = target_name,
    FacetPanel = facet_label_str,
    SortOrder = i 
  )
  plot_list_df[[target_name]] <- temp_df
}

# 合并所有数据
plot_df_final <- do.call(rbind, plot_list_df)

# 重新排序 Factor
plot_df_final <- plot_df_final %>% arrange(SortOrder)
unique_panels <- unique(plot_df_final$FacetPanel)
plot_df_final$FacetPanel <- factor(plot_df_final$FacetPanel, levels = unique_panels)

# --- 4. 绘图 ---
p <- ggplot(plot_df_final, aes(x = U1, y = U2)) +
  
  # A. 负样本
  geom_point(data = ~ subset(.x, PlotStatus == "Other"),
             color = morandi_grey, size = 0.4, alpha = 0.25) +
  
  # B. 正样本
  geom_point(data = ~ subset(.x, PlotStatus == "Target"),
             aes(color = RealLabel), 
             size = 1.2, alpha = 0.9, stroke = 0.15) +
  
  # C. 香槟金密度层
  stat_density_2d(
    data = ~ subset(.x, PlotStatus == "Target"),
    aes(alpha = after_stat(nlevel)), 
    geom = "polygon",
    fill = core_champagne,
    color = NA,
    bins = 10,
    h = c(1.5, 1.5)
  ) +
  
  # D. 密度轮廓线
  stat_density_2d(
    data = ~ subset(.x, PlotStatus == "Target"),
    geom = "path",
    color = "#B8A37F",
    size = 0.25,
    bins = 10,
    h = c(1.5, 1.5),
    alpha = 0.8 
  ) +
  
  scale_color_manual(values = morandi_colors) +
  scale_alpha_continuous(
    limits = c(0.23, 1.0), 
    range = c(0.2, 0.7), 
    oob = scales::squish, 
    guide = "none"
  ) +
  
  facet_wrap(~ FacetPanel, ncol = 4, scales = "free") +
  
  theme_minimal() +
  labs(
    title = "Methylation Feature Distribution (Multi-label)",
    # subtitle = paste0("Metrics: Silhouette Coefficient (SC) & k-NN Purity (k=", k_neighbors_purity, ")"),
    x = "UMAP Dim 1", 
    y = "UMAP Dim 2"
  ) + 
  theme(
    panel.grid = element_blank(),
    plot.background = element_rect(fill = "#F9F8F7", color = NA),
    panel.background = element_rect(fill = "#F5F4F2", color = NA),
    strip.text = element_text(face = "bold", size = 10, color = "#4A4A4A", lineheight = 1.2),
    strip.background = element_rect(fill = "#E8E6E3", color = "#C8C4C0", linewidth = 0.4),
    axis.text = element_blank(),
    axis.ticks = element_blank(),
    panel.border = element_rect(color = "grey92", fill = NA, linewidth = 0.3),
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    legend.position = "none"
  )

print(p)
ggsave("png/UMAP_MultiLabel_Metrics.pdf", p, width = 16, height = 12, device = cairo_pdf)

<U+539F><U+59CB><U+7EF4><U+5EA6><U+68C0><U+67E5>:
<U+7279><U+5F81><U+77E9><U+9635>: 203993 <U+884C>
<U+6807><U+7B7E><U+77E9><U+9635>: 203993 <U+884C>
<U+2705> <U+7EF4><U+5EA6><U+5BF9><U+9F50><U+68C0><U+67E5><U+901A><U+8FC7><U+3002>
[1] "<U+5F00><U+59CB><U+5904><U+7406> 12 <U+4E2A><U+7C7B><U+522B>... target_n = Inf"


In [ ]:
install.packages("cluster")